# Energy resolution

## Physics

The energy resolution of the $Q_{\beta\beta}$ peak is one of the key parameters of the analysis.

The resolution in fact defines the width of the RoI and therefore the possible number of background events.

## Analysis

In [41]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### Importing modules

In [42]:
import numpy  as np
import tables as tb
import pandas as pd
import matplotlib.pyplot as plt

import scipy.constants as constants

import warnings
warnings.filterwarnings('ignore')

In [43]:
import os, sys
from pathlib import Path

# Find the project root: honours FANAL_ROOT env-var, otherwise walks up from cwd
_env = os.environ.get('FANAL_ROOT') or os.environ.get('USCFANALDIR')
if _env and Path(_env, 'data').is_dir():
    rootpath = str(Path(_env).resolve())
else:
    rootpath = str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'data').is_dir() and (p / 'ana').is_dir()))
if rootpath not in sys.path:
    sys.path.insert(0, rootpath)
print('Fanal root : ', rootpath)

Fanal root :  /home/daniel/GitHub/Grupo_beta_DM


In [44]:
import core.pltext  as pltext   # extensions for plotting histograms
import core.hfit    as hfit     # extension to fit histograms
import core.utils   as ut       # generic utilities
import ana.fanal    as fn       # analysis functions specific to fanal
pltext.style()

### Main parameters

In the next cell we define the main parameters of the selection

In [45]:
from collpars import collaboration as coll
print('Collaboration             : {:s}'.format(coll))

Collaboration             : new_beta


## Access the data

In the next cell we read the data (in a *h5* format), and access the simulated data of the backgrounds and the signal. 

We output the number of events in the simulated samples

In [46]:
# set the path to the data directory and filenames
dirpath = rootpath+'/data/'
filename = 'fanal_' + coll + '.h5'
print('Data path and filename : ', dirpath + filename)

# access the simulated data (DataFrames) for the different samples (Bi, Tl, bb) located in the data file
mcbi = pd.read_hdf(dirpath + filename, key = 'mc/bi214').fillna(0.)
#mctl = pd.read_hdf(dirpath + filename, key = 'mc/tl208')
#mcbb = pd.read_hdf(dirpath + filename, key = 'mc/bb0nu')

# set the names of the samples
mc_samples         = [mcbi,] # list of the mc DFs
sample_names       = ['Bi',]
sample_names_latex = [ r'$^{214}$Bi',] # str names of the mc samples

for i, mc in enumerate(mc_samples):
    print('MC Sample {:s}, number of simulated events = {:d}'.format(sample_names[i], len(mc)))

Data path and filename :  /home/daniel/GitHub/Grupo_beta_DM/data/fanal_new_beta.h5
MC Sample Bi, number of simulated events = 60184


## Estimate the energy resolution 

The energy resolution is a crucial parameter in this analysis.

We fit the energy distribution of the $^{214}\mathrm{Bi}$ peak to a Gaussian on top of a pedestal modelled as a straight line.

Give the relative energy resolution as the percentage of $\sigma/E$ or FWHM/E (Full Width at Half Maximum).

*Exercise:* Compute the energy resolution of the other MC samples: $^{208}\mathrm{Tl}$ and the $\beta\beta0\nu$.


In [ ]:
# Fit the energy distribution of each MC sample to a Gaussian + line model.
# Use hfit.hfitres() with the fgausline function, e.g.:
#   par, cov, _, _ = hfit.hfitres(mc.E[sel], bins, range=sel_erange, fun=hfit.fgausline)
#
# Extract sigma from the fit parameters and compute the energy resolution:
#   resolution = sigma / E_peak  (or FWHM / E_peak, where FWHM = 2.355 * sigma)
#   collaboration
# Store the results in a list called 'resolutions',
# where resolutions[i] = (sigma_i, ...) for each sample.
# YOUR CODE HERE



### Notation-to-code reference

| Math | Python variable | Description |
|------|-----------------|-------------|
| $\sigma_\mathrm{Bi}$ | `sigma_Bi` | Energy resolution of $^{214}$Bi (keV) |
| $\sigma_\mathrm{Tl}$ | `sigma_Tl` | Energy resolution of $^{208}$Tl (keV) |
| $\sigma_{\beta\beta}$ | `sigma_bb` | Energy resolution of $\beta\beta0\nu$ (keV) |

In [48]:
import ana.fanal_display as fdisp

# Check your values before writing to collpars.py
fdisp.display_collpars([
    (r'\sigma_\mathrm{' + sample + '}', 'sigma_' + sample, resolutions[i][0], '.2e')
    for i, sample in enumerate(sample_names)
])

NameError: name 'resolutions' is not defined

### Write out

We write the relevant python variables into the file *collpars.py*, which contains the relevant parameters of the analysis and which will be used in the following notebooks.

We write out into the parameter file:

  * The energy resolution of each sample

In [ ]:
write = True
collpar_filename = "collpars.py"
if (write):
    of = open(collpar_filename, 'a')
    for i, sample in enumerate(sample_names):
        sigma = resolutions[i][0]
        of.write('sigma_'+sample+'        = {:1.2e} # keV'.format(sigma)+'\n')
    of.close()